In [2]:
import time
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

print("=== DEPLOYING RAG RETRIEVAL ENGINE ===")

# STEP 1: AUTHENTICATE AND CONNECT
PINECONE_API_KEY = "<Enter your APIs>"

# Initialize the cloud client controller
pc = Pinecone(api_key=PINECONE_API_KEY)

# Define the index we created in our last lab module
index_name = "infra-cost-index"

# Connect to the live cloud-native index endpoint
index = pc.Index(index_name)
print(f"Successfully established connection to cloud index: '{index_name}'")

=== DEPLOYING RAG RETRIEVAL ENGINE ===
Successfully established connection to cloud index: 'infra-cost-index'


In [3]:
# STEP 2: LOAD THE LOCAL EMBEDDING PIPELINE
print("Loading local semantic vector transformer...")
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading local semantic vector transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
# STEP 3: CONSTRUCT THE SEMANTIC QUERY
user_query = "Where do we store our data and how much space is allocated?"
print(f"\nUser Input Query:\n -> '{user_query}'")


# STEP 4: VECTORIZE THE QUESTION
print("Computing vector coordinates for the query...")
query_embedding = model.encode(user_query).tolist()


User Input Query:
 -> 'Where do we store our data and how much space is allocated?'
Computing vector coordinates for the query...


In [5]:
# STEP 5: EXECUTE THE SIMILARITY SEARCH
print("Streaming query vector to Pinecone for Cosine Similarity matching...")

# index.query runs high-speed vector navigation across the cloud cluster
search_results = index.query(
    vector=query_embedding,
    top_k=1,
    include_metadata=True
)

Streaming query vector to Pinecone for Cosine Similarity matching...


In [6]:
# STEP 6: PARSE AND PRINT THE OUTPUT
print("\n=== LIVE RETRIEVAL ENGINE SEARCH RESULTS ===")
matches = search_results['matches']

if len(matches) == 0:
    print("ALERT: No relevant matches found in the vector index topology.")
else:
    best_match = matches[0]
    print(f"Match Record ID:    {best_match['id']}")
    print(f"Similarity Score:   {best_match['score']:.4f} ({best_match['score']*100:.1f}% match)")
    
    # Extracting the raw text metadata we saved during ingestion
    retrieved_text = best_match['metadata']['raw_text']
    print(f"\n[RETRIEVED DOCUMENT SNIPPET]:\n -> \"{retrieved_text}\"")

print("===========================================================")
print("SUCCESS: Meaning-based retrieval process is complete!")


=== LIVE RETRIEVAL ENGINE SEARCH RESULTS ===
Match Record ID:    vector_id_001
Similarity Score:   0.4667 (46.7% match)

[RETRIEVED DOCUMENT SNIPPET]:
 -> "Our persistent storage allocation is 200 GB General Purpose EBS costing $16.00."
SUCCESS: Meaning-based retrieval process is complete!
